# Anote 1A — Unified Dataset

Combines the code-generation and bug-fixing evaluator files into one dataset (working-order step 3: integration).

**Inputs**
- `data/code_generation/code_generation_eval.jsonl` (from `code_generation_EDA_pipeline.ipynb`)
- `data/bug_fixes/bug_fixing_eval.jsonl` (from `bug_fixes_EDA_pipeline.ipynb`)
- each task's `*_full.parquet`, only to fill the `metadata` column

**Output:** `data/unified/unified_eval.jsonl`

**Steps**
1. Add a `metadata` column: task-specific details, one dict per row
2. Keep only the columns both task types share (everything else stays in each task's `*_full.parquet`)
3. Concatenate and check every row is tagged with its `task_type`
4. Drop malformed rows
5. Deduplicate
6. Export

## 1. Setup

In [2]:
import json
import re
from pathlib import Path

import pandas as pd

# Repo root = first folder at or above the working directory that contains Evaluator/evaluator.py
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Evaluator" / "evaluator.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repo (Colab: git clone the repo and %cd into notebooks/).")

DATA_DIR = REPO_ROOT / "data"
INPUTS = {
    "code_generation": DATA_DIR / "code_generation" / "code_generation_eval.jsonl",
    "bug_fixing": DATA_DIR / "bug_fixes" / "bug_fixing_eval.jsonl",
}
FULL_FILES = {
    "code_generation": DATA_DIR / "code_generation" / "code_generation_full.parquet",
    "bug_fixing": DATA_DIR / "bug_fixes" / "bug_fixing_full.parquet",
}
PROCESSED_DIR = DATA_DIR / "unified"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] Environment initialized. Directories verified.")

[INFO] Environment initialized. Directories verified.


## 2. Load both task types
`dtype=False` keeps `tests` as real lists (and ids as strings).

In [3]:
frames = {task: pd.read_json(path, lines=True, dtype=False) for task, path in INPUTS.items()}

for task, df in frames.items():
    print(f"[INFO] {task}: {len(df)} rows, columns = {list(df.columns)}")

[INFO] code_generation: 672 rows, columns = ['id', 'task_type', 'source', 'prompt', 'reference', 'tests', 'test_format', 'entry_point']
[INFO] bug_fixing: 204 rows, columns = ['id', 'task_type', 'source', 'prompt', 'reference', 'tests', 'test_format', 'entry_point']


## 3. Add per-task metadata
Every row gets a `metadata` dict, so both task types still share the same columns. The keys depend on the task type:

| `task_type` | `metadata` keys |
|---|---|
| `code_generation` | `difficulty` (LiveCodeBench `medium` / `hard`, else `null`) |
| `bug_fixing` | `buggy_code`, `bug_diff` (the reference fix as a diff), `num_changed_lines`, `bug_type` (HumanEval-Fix only) |

**Never send `metadata` to the model.** `bug_diff` is the answer. The model only ever sees `prompt`. The metadata is for scoring and analysis:
- Diff the model's output against `buggy_code` to see which lines it changed, then compare with `bug_diff`: did it fix the right place, even when tests fail?
- Split results by `num_changed_lines`: 180 of 204 bug fixes are one-line (all of QuixBugs, 140 of 164 HumanEval-Fix). This is the column that records it.

In [4]:
METADATA_COLS = {
    "code_generation": ["difficulty"],
    "bug_fixing": ["buggy_code", "bug_diff", "num_changed_lines", "bug_type"],
}

for task, df in frames.items():
    full = pd.read_parquet(FULL_FILES[task]).set_index("id")[METADATA_COLS[task]]
    missing = set(df["id"]) - set(full.index)
    assert not missing, f"{task}: ids missing from {FULL_FILES[task].name}: {sorted(missing)[:5]}"

    # NaN -> None so missing values export as JSON null
    full = full.astype(object).where(full.notna(), None)
    records = full.to_dict("index")
    df["metadata"] = df["id"].map(records)

print(frames["bug_fixing"]["metadata"].iloc[0])

{'buggy_code': 'def bitcount(n):\n    count = 0\n    while n:\n        n ^= n - 1\n        count += 1\n    return count', 'bug_diff': '--- buggy\n+++ fixed\n@@ -4 +4 @@\n-        n ^= n - 1\n+        n &= n - 1', 'num_changed_lines': 1, 'bug_type': None}


## 4. Keep only the shared columns
Any column that isn't in **both** files is dropped. The column order follows the code-generation file.

In [5]:
shared_cols = [c for c in frames["code_generation"].columns if all(c in df.columns for df in frames.values())]
for task, df in frames.items():
    dropped = [c for c in df.columns if c not in shared_cols]
    print(f"{task}: dropping {dropped if dropped else 'nothing'}")

df_unified = pd.concat([df[shared_cols] for df in frames.values()], ignore_index=True)

# Every row must be tagged with one of the known task types
assert set(df_unified["task_type"]) == set(INPUTS), f"Unexpected task_type values: {set(df_unified['task_type'])}"
print(f"\nShared columns: {shared_cols}")
print(f"Total rows: {len(df_unified)}")

code_generation: dropping nothing
bug_fixing: dropping nothing

Shared columns: ['id', 'task_type', 'source', 'prompt', 'reference', 'tests', 'test_format', 'entry_point', 'metadata']
Total rows: 876


## 5. Drop malformed rows
A row is malformed if it can't be sent to a model or scored:
- `id`, `task_type`, `source` or `prompt` is missing / empty
- `tests` is not a non-empty list
- `reference` is empty, **except** for LiveCodeBench, which ships no reference solutions by design (it is scored on tests only)

In [6]:
NO_REFERENCE_SOURCES = {"livecodebench"}


def is_blank(value) -> bool:
    return not isinstance(value, str) or not value.strip()


def malformed_reason(row) -> str:
    for col in ["id", "task_type", "source", "prompt"]:
        if is_blank(row[col]):
            return f"empty {col}"
    if not isinstance(row["tests"], list) or not row["tests"]:
        return "no tests"
    if is_blank(row["reference"]) and row["source"] not in NO_REFERENCE_SOURCES:
        return "empty reference"
    return ""


df_unified["malformed_reason"] = df_unified.apply(malformed_reason, axis=1)
malformed = df_unified[df_unified["malformed_reason"] != ""]
if len(malformed):
    print("Dropped malformed rows:\n" + malformed[["id", "malformed_reason"]].to_string(index=False))
df_unified = df_unified[df_unified["malformed_reason"] == ""].drop(columns="malformed_reason").reset_index(drop=True)
print(f"Dropped {len(malformed)} malformed rows. Remaining: {len(df_unified)}")

Dropped 0 malformed rows. Remaining: 876


## 6. Deduplicate
- `id` must be unique across the whole dataset.
- **Same task, reworded prompt:** two rows of the same `task_type` with the same reference solution (whitespace ignored) are the same problem. The first one is kept. (Example: MBPP 758 and 779 have the same reference; only the prompt wording and one space in a test differ.)
- **Identical prompts** within a task type are also dropped.

Not counted as duplicates: HumanEval (code generation) and HumanEval-Fix (bug fixing) share problems and references, but they are different tasks with different prompts, so both are kept.

In [7]:
assert df_unified["id"].is_unique, "Duplicate ids across task types"


def norm(s) -> str:
    return re.sub(r"\s+", " ", s).strip() if isinstance(s, str) else ""


df_unified["_ref_key"] = df_unified["reference"].map(norm)
df_unified["_prompt_key"] = df_unified["prompt"].map(norm)

has_ref = df_unified["_ref_key"] != ""
dup_problem = has_ref & df_unified.duplicated(["task_type", "_ref_key"])
dup_prompt = df_unified.duplicated(["task_type", "_prompt_key"])
is_dup = dup_problem | dup_prompt

if is_dup.any():
    print("Dropped duplicates:\n" + df_unified.loc[is_dup, ["id", "task_type", "source"]].to_string(index=False))
df_unified = df_unified[~is_dup].drop(columns=["_ref_key", "_prompt_key"]).reset_index(drop=True)
print(f"Dropped {is_dup.sum()} duplicates. Remaining: {len(df_unified)}")

Dropped duplicates:
      id       task_type source
mbpp-779 code_generation   mbpp
Dropped 1 duplicates. Remaining: 875


## 7. Export

In [8]:
unified_path = PROCESSED_DIR / "unified_eval.jsonl"
df_unified.to_json(unified_path, orient="records", lines=True)

# Round-trip: tests must come back as lists
check = pd.read_json(unified_path, lines=True, dtype=False)
assert len(check) == len(df_unified) and isinstance(check.loc[0, "tests"], list)
assert check["metadata"].map(lambda m: isinstance(m, dict)).all(), "Every row needs a metadata dict"

print("=== Unified Dataset ===")
print(df_unified.groupby(["task_type", "source"]).size().to_string())
print(f"Total tasks: {len(df_unified)}")
print(f"\n[SUCCESS] Saved to {unified_path.resolve()} ({len(shared_cols)} columns)")

=== Unified Dataset ===
task_type        source       
bug_fixing       humanevalfix     164
                 quixbugs          40
code_generation  humaneval        164
                 livecodebench    131
                 mbpp             376
Total tasks: 875

[SUCCESS] Saved to /Users/diadana/Documents/GitHub/Anote-1A-building-high-quality-datasets-and-evaluation-frameworks-for-ai-coding-assistants/data/unified/unified_eval.jsonl (9 columns)
